# 04 BM25
- **Estudiante**: J. L. Andino
- **Fecha de Entrega:** Viernes 13 de noviembre de 2026
- **Curso:** GR1CC
- **Carrera:** Computación
- **Docente:** Prof. Iván Carrera
## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
# ============ Celda 1: Corpus, tokens, DF e IDF (Sección 1) ============
import math
import re
import sys
import unicodedata
from abc import ABC, abstractmethod
from collections import Counter
from dataclasses import dataclass

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg
from scipy import sparse

nltk.download("gutenberg", quiet=True)


class TextNormalizer:
    """Preprocesamiento de la práctica anterior: minúsculas -> NFKD sin diacríticos -> tokens [a-z0-9]+."""
    _TOKEN_RE = re.compile(r"[a-z0-9]+")

    def normalize(self, text: str) -> str:
        text = unicodedata.normalize("NFKD", text.lower())
        return text.encode("ascii", "ignore").decode("ascii")

    def tokenize(self, text: str) -> list[str]:
        return [sys.intern(t) for t in self._TOKEN_RE.findall(self.normalize(text))]


TITLES = {
    "austen-emma.txt": "Emma",
    "austen-persuasion.txt": "Persuasion",
    "austen-sense.txt": "Sense and Sensibility",
    "bible-kjv.txt": "The King James Bible",
    "blake-poems.txt": "Poems",
    "bryant-stories.txt": "Stories to Tell to Children",
    "burgess-busterbrown.txt": "The Adventures of Buster Bear",
    "carroll-alice.txt": "Alice's Adventures in Wonderland",
    "chesterton-ball.txt": "The Ball and the Cross",
    "chesterton-brown.txt": "The Wisdom of Father Brown",
    "chesterton-thursday.txt": "The Man Who Was Thursday",
    "edgeworth-parents.txt": "The Parent's Assistant",
    "melville-moby_dick.txt": "Moby Dick",
    "milton-paradise.txt": "Paradise Lost",
    "shakespeare-caesar.txt": "Julius Caesar",
    "shakespeare-hamlet.txt": "Hamlet",
    "shakespeare-macbeth.txt": "Macbeth",
    "whitman-leaves.txt": "Leaves of Grass",
}


def load_corpus_table(normalizer: TextNormalizer) -> pd.DataFrame:
    """Tabla Identificador | Título | Texto, más la columna `tokens` (lista de tokens normalizados)."""
    table = pd.DataFrame({
        "Identificador": [str(f) for f in gutenberg.fileids()],        # IDs siempre como cadenas
        "Título": [TITLES[f] for f in gutenberg.fileids()],
        "Texto": [gutenberg.raw(f) for f in gutenberg.fileids()],
    })
    table["tokens"] = table["Texto"].map(normalizer.tokenize)
    return table


@dataclass(frozen=True)
class CollectionStats:
    """Estadísticas de la colección: se calculan una vez y quedan de solo lectura (no cambian entre consultas)."""
    doc_ids: tuple[str, ...]
    titles: tuple[str, ...]
    vocabulary: tuple[str, ...]
    term_index: dict[str, int]
    tf: sparse.csr_matrix          # f(t,d): documentos x términos, dispersa
    doc_len: np.ndarray            # |d|
    df: np.ndarray                 # df(t), alineado con `vocabulary`

    def __post_init__(self):
        for arr in (self.doc_len, self.df, self.tf.data):
            arr.setflags(write=False)

    @property
    def n_docs(self) -> int:
        return len(self.doc_ids)

    @property
    def avgdl(self) -> float:
        return float(self.doc_len.mean())

    @classmethod
    def from_table(cls, table: pd.DataFrame) -> "CollectionStats":
        vocabulary = tuple(sorted({t for tokens in table["tokens"] for t in tokens}))
        term_index = {t: i for i, t in enumerate(vocabulary)}
        rows, cols, data = [], [], []
        for row, tokens in enumerate(table["tokens"]):
            for term, freq in Counter(tokens).items():
                rows.append(row), cols.append(term_index[term]), data.append(freq)
        tf = sparse.csr_matrix((data, (rows, cols)), shape=(len(table), len(vocabulary)), dtype=np.float64)
        return cls(
            doc_ids=tuple(table["Identificador"]),
            titles=tuple(table["Título"]),
            vocabulary=vocabulary,
            term_index=term_index,
            tf=tf,
            doc_len=table["tokens"].map(len).to_numpy(dtype=np.float64),
            df=np.diff(tf.tocsc().indptr).astype(np.float64),      # documentos con f(t,d) > 0
        )


def idf_tfidf(stats: CollectionStats) -> np.ndarray:
    """IDF de la práctica anterior: ln(N / df) + 1."""
    return np.log(stats.n_docs / stats.df) + 1


def idf_bm25(stats: CollectionStats) -> np.ndarray:
    """IDF de BM25 (variante Lucene/Robertson): ln(1 + (N - df + 0.5) / (df + 0.5)); siempre positivo."""
    return np.log1p((stats.n_docs - stats.df + 0.5) / (stats.df + 0.5))


normalizer = TextNormalizer()
corpus = load_corpus_table(normalizer)
stats = CollectionStats.from_table(corpus)

print(f"Documentos: {stats.n_docs} | Vocabulario: {len(stats.vocabulary):,} | "
      f"|d| mínima: {stats.doc_len.min():,.0f} | media (avgdl): {stats.avgdl:,.1f} | máxima: {stats.doc_len.max():,.0f}")
display(corpus[["Identificador", "Título", "tokens"]].assign(
    longitud=lambda t: t["tokens"].map(len), tokens=lambda t: t["tokens"].map(lambda x: x[:6])))

SAMPLE_TERMS = ["the", "sea", "whale", "ahab", "knightley"]
display(pd.DataFrame({
    "df": [int(stats.df[stats.term_index[t]]) for t in SAMPLE_TERMS],
    "IDF TF-IDF": [idf_tfidf(stats)[stats.term_index[t]] for t in SAMPLE_TERMS],
    "IDF BM25": [idf_bm25(stats)[stats.term_index[t]] for t in SAMPLE_TERMS],
}, index=SAMPLE_TERMS).round(4))

Documentos: 18 | Vocabulario: 41,838 | |d| mínima: 6,936 | media (avgdl): 122,178.2 | máxima: 854,046


,Identificador,Título,tokens,longitud
0,austen-emma.txt,Emma,"[emma, by, jane, austen, 1816, volume]",161980
1,austen-persuasion.txt,Persuasion,"[persuasion, by, jane, austen, 1818, chapter]",84167
2,austen-sense.txt,Sense and Sensibility,"[sense, and, sensibility, by, jane, austen]",120787
3,bible-kjv.txt,The King James Bible,"[the, king, james, bible, the, old]",854046
4,blake-poems.txt,Poems,"[poems, by, william, blake, 1789, songs]",6936
5,bryant-stories.txt,Stories to Tell to Children,"[stories, to, tell, to, children, by]",46699
6,burgess-busterbrown.txt,The Adventures of Buster Bear,"[the, adventures, of, buster, bear, by]",16359
7,carroll-alice.txt,Alice's Adventures in Wonderland,"[alice, s, adventures, in, wonderland, by]",27336
8,chesterton-ball.txt,The Ball and the Cross,"[the, ball, and, the, cross, by]",82865
9,chesterton-brown.txt,The Wisdom of Father Brown,"[the, wisdom, of, father, brown, by]",73288


,df,IDF TF-IDF,IDF BM25
the,18,1.0000,0.0267
sea,16,1.1178,0.1411
whale,6,2.0986,1.0726
ahab,2,3.1972,2.0281
knightley,1,3.8904,2.5390


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
# ============ Celda 2: Consultas, scorers y ranking (Sección 2) ============
K1, B = 1.2, 0.75
TOP_K = 10


@dataclass(frozen=True)
class QuerySpec:
    query_id: str
    need: str     # qué busca el usuario
    text: str


QUERIES = [
    QuerySpec("q1", "Pasajes sobre la ballena (término de frecuencia media)", "whale"),
    QuerySpec("q2", "Palabra ubicua: no discrimina entre documentos", "the"),
    QuerySpec("q3", "Un personaje concreto y muy raro (df=1)", "knightley"),
    QuerySpec("q4", "La caza de la ballena blanca y el capitán obsesionado", "captain ahab white whale sea ship"),
    QuerySpec("q5", "Novelas de costumbres: matrimonio y vida social", "love marriage gentleman lady"),
    QuerySpec("q6", "Creación y caída del hombre", "god heaven adam eve serpent"),
    QuerySpec("q7", "Tragedias de asesinato y realeza", "murder ghost king queen"),
    QuerySpec("q8", "Mezcla de palabras muy frecuentes y de frecuencia media", "the king and queen"),
]


class Scorer(ABC):
    """Contrato común: toda técnica de ranking aporta contribuciones por término; el score es su suma."""

    def __init__(self, stats: CollectionStats):
        self._stats = stats

    @abstractmethod
    def term_contributions(self, query: Counter) -> np.ndarray:
        """Matriz documentos x términos de la consulta (columnas en el orden de `query`)."""

    def scores(self, query: Counter) -> np.ndarray:
        return self.term_contributions(query).sum(axis=1)


class TfidfCosineScorer(Scorer):
    """TF-IDF + coseno sobre matriz dispersa. La consulta usa el vocabulario e IDF del corpus."""

    def __init__(self, stats: CollectionStats):
        super().__init__(stats)
        self._idf = idf_tfidf(stats)
        self._matrix = (stats.tf @ sparse.diags(self._idf)).tocsr()
        self._doc_norm = np.sqrt(self._matrix.multiply(self._matrix).sum(axis=1)).A1

    def term_contributions(self, query: Counter) -> np.ndarray:
        idx = [self._stats.term_index[t] for t in query]
        if not idx:
            return np.zeros((self._stats.n_docs, 0))
        q_weights = np.array([query[t] for t in query]) * self._idf[idx]
        q_norm = np.linalg.norm(q_weights)
        dots = self._matrix[:, idx].multiply(q_weights).toarray()        # q_i * d_i por término
        return dots / (q_norm * self._doc_norm[:, None])


class BM25Scorer(Scorer):
    """BM25: idf(t) * f(k1+1) / (f + k1(1-b+b|d|/avgdl)), sumado sobre los términos distintos de la consulta."""

    def __init__(self, stats: CollectionStats, k1: float = K1, b: float = B):
        super().__init__(stats)
        self.k1, self.b = k1, b
        self._idf = idf_bm25(stats)
        self._length_norm = k1 * (1 - b + b * stats.doc_len / stats.avgdl)   # K_d, uno por documento
        self._tf_csc = stats.tf.tocsc()

    def term_contributions(self, query: Counter) -> np.ndarray:
        contributions = np.zeros((self._stats.n_docs, len(query)))
        for j, term in enumerate(query):
            t = self._stats.term_index[term]
            f = self._tf_csc[:, t].toarray().ravel()
            contributions[:, j] = self._idf[t] * f * (self.k1 + 1) / (f + self._length_norm)
        return contributions


@dataclass(frozen=True)
class RankedDoc:
    position: int
    doc_id: str
    title: str
    score: float


def analyze_query(text: str, stats: CollectionStats) -> Counter:
    """Mismo preprocesamiento que los documentos; descarta términos fuera del vocabulario."""
    return Counter(t for t in normalizer.tokenize(text) if t in stats.term_index)


def rank(scores: np.ndarray, stats: CollectionStats, top_k: int | None = TOP_K) -> list[RankedDoc]:
    """Score descendente, desempate por doc_id (cadena) ascendente; solo scores positivos."""
    order = sorted(
        (i for i in range(stats.n_docs) if scores[i] > 0),
        key=lambda i: (-round(float(scores[i]), 12), stats.doc_ids[i]),   # el redondeo evita falsos desempates por ruido float
    )
    return [RankedDoc(pos, stats.doc_ids[i], stats.titles[i], float(scores[i]))
            for pos, i in enumerate(order[:top_k], start=1)]


tfidf_scorer = TfidfCosineScorer(stats)
bm25_scorer = BM25Scorer(stats, K1, B)

rankings = {}
for spec in QUERIES:
    query = analyze_query(spec.text, stats)
    rankings[spec.query_id] = (rank(tfidf_scorer.scores(query), stats), rank(bm25_scorer.scores(query), stats))

# Comprobación: consulta sin términos del vocabulario -> dos listas vacías
empty_query = analyze_query("zxqwv qwertyuiop", stats)
assert rank(tfidf_scorer.scores(empty_query), stats) == [] and rank(bm25_scorer.scores(empty_query), stats) == []

# Justificación de las consultas con df
display(pd.DataFrame({
    "Consulta": [s.query_id for s in QUERIES],
    "Qué busca el usuario": [s.need for s in QUERIES],
    "Texto": [s.text for s in QUERIES],
    "Términos": [len(analyze_query(s.text, stats)) for s in QUERIES],
    f"df de cada término (N={stats.n_docs})": [
        ", ".join(f"{t}={int(stats.df[stats.term_index[t]])}" for t in analyze_query(s.text, stats)) for s in QUERIES],
    "Docs TF-IDF": [len(rankings[s.query_id][0]) for s in QUERIES],
    "Docs BM25": [len(rankings[s.query_id][1]) for s in QUERIES],
}).set_index("Consulta"))

,Qué busca el usuario,Texto,Términos,df de cada término (N=18),Docs TF-IDF,Docs BM25
Consulta,,,,,,
q1,Pasajes sobre la ballena (término de frecuenci...,whale,1,whale=6,6,6
q2,Palabra ubicua: no discrimina entre documentos,the,1,the=18,10,10
q3,Un personaje concreto y muy raro (df=1),knightley,1,knightley=1,1,1
q4,La caza de la ballena blanca y el capitán obse...,captain ahab white whale sea ship,6,"captain=9, ahab=2, white=16, whale=6, sea=16, ...",10,10
q5,Novelas de costumbres: matrimonio y vida social,love marriage gentleman lady,4,"love=15, marriage=12, gentleman=12, lady=15",10,10
q6,Creación y caída del hombre,god heaven adam eve serpent,5,"god=16, heaven=13, adam=7, eve=7, serpent=9",10,10
q7,Tragedias de asesinato y realeza,murder ghost king queen,4,"murder=9, ghost=9, king=15, queen=15",10,10
q8,Mezcla de palabras muy frecuentes y de frecuen...,the king and queen,4,"the=18, king=15, and=18, queen=15",10,10


**Justificación de las consultas con $df$** (N = 18 documentos)

| Consulta | Longitud | $df$ de sus términos | Qué pone a prueba |
|---|---:|---|---|
| q1 `whale` | 1 | 6 | Un término de frecuencia media con $f(t,d)$ muy desigual entre libros |
| q2 `the` | 1 | 18 | Término presente en todos los documentos: IDF TF-IDF = 1.00 y IDF BM25 = 0.027 |
| q3 `knightley` | 1 | 1 | El término más raro posible: un único documento con score positivo |
| q4 `captain ahab white whale sea ship` | 6 | 9, 2, 16, 6, 16, 13 | Mezcla de términos raros (`ahab`) y comunes (`white`, `sea`) |
| q5 `love marriage gentleman lady` | 4 | 15, 12, 12, 15 | Términos frecuentes de un mismo campo semántico |
| q6 `god heaven adam eve serpent` | 5 | 16, 13, 7, 7, 9 | Consulta larga con $df$ muy dispares |
| q7 `murder ghost king queen` | 4 | 9, 9, 15, 15 | Dos términos medios y dos frecuentes |
| q8 `the king and queen` | 4 | 18, 15, 18, 15 | Palabras vacías mezcladas con términos de contenido |

Solo q1 y q3 devuelven menos de diez documentos (6 y 1), porque solo esos documentos contienen el término. Una consulta sin términos del vocabulario produce dos listas vacías (comprobado con `assert`). Las estadísticas de la colección ($N$, $|d|$, avgdl, $df$) son de solo lectura, así que no cambian entre consultas.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
# ============ Celda 3: Comparación de rankings (Sección 3) ============
def label(doc: RankedDoc | None) -> str:
    return f"{doc.doc_id} · {doc.title}" if doc else ""


def comparison_table(tfidf: list[RankedDoc], bm25: list[RankedDoc]) -> pd.DataFrame:
    rows = []
    for i in range(max(len(tfidf), len(bm25))):
        t = tfidf[i] if i < len(tfidf) else None
        b = bm25[i] if i < len(bm25) else None
        rows.append({
            "Posición": i + 1,
            "ID y título TF-IDF": label(t),
            "Coseno": round(t.score, 6) if t else "",
            "ID y título BM25": label(b),
            "Score BM25": round(b.score, 6) if b else "",
        })
    return pd.DataFrame(rows).set_index("Posición")


def overlap_summary(tfidf: list[RankedDoc], bm25: list[RankedDoc]) -> dict:
    ids_t, ids_b = [d.doc_id for d in tfidf], [d.doc_id for d in bm25]
    return {
        "|T_TFIDF|": len(ids_t),
        "|T_BM25|": len(ids_b),
        "|T_TFIDF ∩ T_BM25|": len(set(ids_t) & set(ids_b)),
        "misma posición": sum(a == b for a, b in zip(ids_t, ids_b)),
        "1.º TF-IDF": ids_t[0] if ids_t else "-",
        "1.º BM25": ids_b[0] if ids_b else "-",
    }


summary = {}
for spec in QUERIES:
    tfidf, bm25 = rankings[spec.query_id]
    summary[spec.query_id] = overlap_summary(tfidf, bm25)
    print(f"\n{spec.query_id}: '{spec.text}'  ->  {summary[spec.query_id]}")
    display(comparison_table(tfidf, bm25))

print("\nResumen de solapamiento por consulta")
display(pd.DataFrame(summary).T)


q1: 'whale'  ->  {'|T_TFIDF|': 6, '|T_BM25|': 6, '|T_TFIDF ∩ T_BM25|': 6, 'misma posición': 4, '1.º TF-IDF': 'melville-moby_dick.txt', '1.º BM25': 'melville-moby_dick.txt'}


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,melville-moby_dick.txt · Moby Dick,0.119448,melville-moby_dick.txt · Moby Dick,2.356130
2,bryant-stories.txt · Stories to Tell to Children,0.006478,bryant-stories.txt · Stories to Tell to Children,2.268494
3,shakespeare-hamlet.txt · Hamlet,0.001520,whitman-leaves.txt · Leaves of Grass,1.893108
4,whitman-leaves.txt · Leaves of Grass,0.000751,shakespeare-hamlet.txt · Hamlet,1.870642
5,chesterton-ball.txt · The Ball and the Cross,0.000491,chesterton-ball.txt · The Ball and the Cross,1.621629
6,bible-kjv.txt · The King James Bible,0.000060,bible-kjv.txt · The King James Bible,0.738118



q2: 'the'  ->  {'|T_TFIDF|': 10, '|T_BM25|': 10, '|T_TFIDF ∩ T_BM25|': 8, 'misma posición': 2, '1.º TF-IDF': 'whitman-leaves.txt', '1.º BM25': 'bible-kjv.txt'}


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,whitman-leaves.txt · Leaves of Grass,0.723419,bible-kjv.txt · The King James Bible,0.058664
2,melville-moby_dick.txt · Moby Dick,0.669966,whitman-leaves.txt · Leaves of Grass,0.058663
3,bryant-stories.txt · Stories to Tell to Children,0.666026,melville-moby_dick.txt · Moby Dick,0.058662
4,chesterton-brown.txt · The Wisdom of Father Brown,0.633626,chesterton-brown.txt · The Wisdom of Father Brown,0.058660
5,blake-poems.txt · Poems,0.611687,chesterton-ball.txt · The Ball and the Cross,0.058659
6,bible-kjv.txt · The King James Bible,0.606516,bryant-stories.txt · Stories to Tell to Children,0.058659
7,chesterton-thursday.txt · The Man Who Was Thur...,0.588002,chesterton-thursday.txt · The Man Who Was Thur...,0.058658
8,chesterton-ball.txt · The Ball and the Cross,0.582693,edgeworth-parents.txt · The Parent's Assistant,0.058658
9,carroll-alice.txt · Alice's Adventures in Wond...,0.547424,austen-persuasion.txt · Persuasion,0.058654



q3: 'knightley'  ->  {'|T_TFIDF|': 1, '|T_BM25|': 1, '|T_TFIDF ∩ T_BM25|': 1, 'misma posición': 1, '1.º TF-IDF': 'austen-emma.txt', '1.º BM25': 'austen-emma.txt'}


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,austen-emma.txt · Emma,0.098606,austen-emma.txt · Emma,5.564383


... (8 outputs left out)


q8: 'the king and queen'  ->  {'|T_TFIDF|': 10, '|T_BM25|': 10, '|T_TFIDF ∩ T_BM25|': 8, 'misma posición': 2, '1.º TF-IDF': 'bible-kjv.txt', '1.º BM25': 'carroll-alice.txt'}


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,bible-kjv.txt · The King James Bible,0.516294,carroll-alice.txt · Alice's Adventures in Wond...,1.006613
2,bryant-stories.txt · Stories to Tell to Children,0.507164,shakespeare-hamlet.txt · Hamlet,0.984266
3,blake-poems.txt · Poems,0.506074,bryant-stories.txt · Stories to Tell to Children,0.967694
4,whitman-leaves.txt · Leaves of Grass,0.504848,bible-kjv.txt · The King James Bible,0.963279
5,melville-moby_dick.txt · Moby Dick,0.444674,melville-moby_dick.txt · Moby Dick,0.957190
6,milton-paradise.txt · Paradise Lost,0.434641,edgeworth-parents.txt · The Parent's Assistant,0.938622
7,chesterton-brown.txt · The Wisdom of Father Brown,0.429021,milton-paradise.txt · Paradise Lost,0.937089
8,carroll-alice.txt · Alice's Adventures in Wond...,0.412093,chesterton-brown.txt · The Wisdom of Father Brown,0.897949
9,chesterton-ball.txt · The Ball and the Cross,0.409592,chesterton-ball.txt · The Ball and the Cross,0.892723



Resumen de solapamiento por consulta


,|T_TFIDF|,|T_BM25|,|T_TFIDF ∩ T_BM25|,misma posición,1.º TF-IDF,1.º BM25
q1,6,6,6,4,melville-moby_dick.txt,melville-moby_dick.txt
q2,10,10,8,2,whitman-leaves.txt,bible-kjv.txt
q3,1,1,1,1,austen-emma.txt,austen-emma.txt
q4,10,10,8,2,melville-moby_dick.txt,melville-moby_dick.txt
q5,10,10,6,0,blake-poems.txt,austen-persuasion.txt
q6,10,10,9,1,milton-paradise.txt,milton-paradise.txt
q7,10,10,6,2,shakespeare-hamlet.txt,shakespeare-hamlet.txt
q8,10,10,8,2,bible-kjv.txt,carroll-alice.txt


In [4]:
# ============ Celda 3b: Dos documentos cuya posición relativa cambia ============
def full_positions(scores: np.ndarray) -> dict[str, int]:
    return {d.doc_id: d.position for d in rank(scores, stats, top_k=None)}


def find_reordered_pair(spec: QuerySpec) -> tuple[str, str] | None:
    """Par (a, b) del top-10 de ambos métodos, con a sobre b en TF-IDF y bajo b en BM25; el de mayor cambio de distancia."""
    tfidf, bm25 = rankings[spec.query_id]
    pos_t, pos_b = {d.doc_id: d.position for d in tfidf}, {d.doc_id: d.position for d in bm25}
    common = sorted(set(pos_t) & set(pos_b))
    pairs = [(a, b) for a in common for b in common if pos_t[a] < pos_t[b] and pos_b[a] > pos_b[b]]
    if not pairs:
        return None
    return max(pairs, key=lambda p: (pos_b[p[0]] - pos_b[p[1]]) - (pos_t[p[0]] - pos_t[p[1]]))


def term_breakdown(spec: QuerySpec, doc_ids: tuple[str, ...]) -> pd.DataFrame:
    """|d|, f(t,d), df(t) y aporte de cada término a BM25 y al coseno TF-IDF, por documento."""
    query = analyze_query(spec.text, stats)
    c_bm25, c_tfidf = bm25_scorer.term_contributions(query), tfidf_scorer.term_contributions(query)
    rows = []
    for doc_id in doc_ids:
        d = stats.doc_ids.index(doc_id)
        for j, term in enumerate(query):
            t = stats.term_index[term]
            rows.append({
                "doc": f"{doc_id} · {stats.titles[d]}", "|d|": int(stats.doc_len[d]), "término": term,
                "f(t,d)": int(stats.tf[d, t]), "df(t)": int(stats.df[t]),
                "aporte BM25": round(c_bm25[d, j], 4), "aporte coseno": round(c_tfidf[d, j], 4),
            })
    return pd.DataFrame(rows)


candidates = {s.query_id: (s, find_reordered_pair(s)) for s in QUERIES}
display(pd.Series({qid: pair for qid, (_, pair) in candidates.items()}, name="par reordenado (a, b)"))

FOCUS_QUERY = next(s for s in QUERIES if s.query_id == "q4")     # se ajusta tras ver los candidatos
focus_pair = candidates[FOCUS_QUERY.query_id][1]
q = analyze_query(FOCUS_QUERY.text, stats)
pos_t, pos_b = full_positions(tfidf_scorer.scores(q)), full_positions(bm25_scorer.scores(q))
sc_t = dict(zip(stats.doc_ids, tfidf_scorer.scores(q)))
sc_b = dict(zip(stats.doc_ids, bm25_scorer.scores(q)))

print(f"Consulta {FOCUS_QUERY.query_id}: '{FOCUS_QUERY.text}' | avgdl = {stats.avgdl:,.1f}")
display(pd.DataFrame({
    "doc": focus_pair, "|d|": [int(stats.doc_len[stats.doc_ids.index(d)]) for d in focus_pair],
    "pos TF-IDF": [pos_t[d] for d in focus_pair], "coseno": [round(sc_t[d], 4) for d in focus_pair],
    "pos BM25": [pos_b[d] for d in focus_pair], "score BM25": [round(sc_b[d], 4) for d in focus_pair],
}).set_index("doc"))
display(term_breakdown(FOCUS_QUERY, focus_pair))

q1    (shakespeare-hamlet.txt, whitman-leaves.txt)
q2             (bryant-stories.txt, bible-kjv.txt)
q3                                            None
q4          (austen-persuasion.txt, bible-kjv.txt)
q5        (edgeworth-parents.txt, austen-emma.txt)
q6             (bible-kjv.txt, whitman-leaves.txt)
q7     (carroll-alice.txt, melville-moby_dick.txt)
q8            (blake-poems.txt, carroll-alice.txt)
Name: par reordenado (a, b), dtype: object

Consulta q4: 'captain ahab white whale sea ship' | avgdl = 122,178.2


,|d|,pos TF-IDF,coseno,pos BM25,score BM25
doc,,,,,
austen-persuasion.txt,84167,2,0.0242,8,2.7317
bible-kjv.txt,854046,8,0.0042,2,7.6422


,doc,|d|,término,"f(t,d)",df(t),aporte BM25,aporte coseno
0,austen-persuasion.txt · Persuasion,84167,captain,303,9,1.5203,0.0231
1,austen-persuasion.txt · Persuasion,84167,ahab,0,2,0.0000,0.0000
2,austen-persuasion.txt · Persuasion,84167,white,5,16,0.2621,0.0002
3,austen-persuasion.txt · Persuasion,84167,whale,0,6,0.0000,0.0000
4,austen-persuasion.txt · Persuasion,84167,sea,21,16,0.2973,0.0007
5,austen-persuasion.txt · Persuasion,84167,ship,6,13,0.6519,0.0003
6,bible-kjv.txt · The King James Bible,854046,captain,139,9,1.4559,0.0008
7,bible-kjv.txt · The King James Bible,854046,ahab,94,2,4.1696,0.0020
8,bible-kjv.txt · The King James Bible,854046,white,75,16,0.2853,0.0002
9,bible-kjv.txt · The King James Bible,854046,whale,3,6,0.7381,0.0000


**Lectura de los rankings**

- Los dos métodos coinciden en el primer documento en q1, q3, q4, q6 y q7. Difieren en q2 (Leaves of Grass frente a la Biblia), q5 (Poems frente a Persuasion) y q8 (la Biblia frente a Alice).
- Compartir IDs no implica ordenarlos igual: en q5 los dos top 10 comparten 6 IDs y **ninguna posición coincide**. En q6 comparten 9 IDs y solo coincide 1 posición.
- q2 (`the`): con BM25 los diez scores están entre 0.058653 y 0.058664, pegados al techo $(k_1+1)\cdot idf = 0.0587$. La saturación hace que una palabra presente en todos los documentos no discrimine. El coseno sí produce una escala amplia (0.72 a 0.50), pero solo refleja qué parte del vector de cada documento ocupa `the`, no relevancia.

**Par analizado: q4 con Persuasion y la Biblia** (avgdl = 122,178)

| | Persuasion | The King James Bible |
|---|---:|---:|
| $\lvert d\rvert$ | 84,167 (0.69 avgdl) | 854,046 (6.99 avgdl) |
| Posición y score TF-IDF | 2 (coseno 0.0242) | 8 (coseno 0.0042) |
| Posición y score BM25 | 8 (2.7317) | 2 (7.6422) |

- **TF-IDF + coseno.** Persuasion tiene `captain` con $f=303$ y $df=9$. Ese único término explica el 95 % de su coseno (0.0231 de 0.0242), porque TF-IDF crece linealmente con $f$. La Biblia tiene un vector de norma mucho mayor ($\lVert d\rVert \approx 105{,}559$ frente a $\approx 8{,}064$), y el coseno divide entre esa norma. Su aporte de `ahab` ($f=94$, $df=2$) es solo 0.0020.
- **BM25.** El aporte de `captain` se satura: Persuasion con $f=303$ aporta 1.5203 y la Biblia con $f=139$ aporta 1.4559, casi lo mismo. Con `ship` pasa igual ($f=6$ y $f=71$ aportan 0.6519 y 0.6880). La Biblia cubre los 6 términos, incluidos los dos más raros: `ahab` aporta 4.1696 (el mayor aporte individual, con $idf=2.03$) y `whale` 0.7381. Persuasion cubre 4 y le faltan `ahab` y `whale`. La penalización por longitud existe ($K_d \approx 6.59$ para la Biblia y $0.92$ para Persuasion), pero no compensa los dos términos raros.
- Cada método se explica con sus propios scores, y no se pueden comparar entre sí porque sus escalas son distintas: el coseno ordena por alineación angular de vectores, y BM25 suma evidencia por término con saturación y rareza.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [5]:
# ============ Celda 4: Efecto de k1 y b (Sección 4) ============
@dataclass(frozen=True)
class BM25Config:
    name: str
    k1: float
    b: float


CONFIGS = [
    BM25Config("Base", 1.2, 0.75),
    BM25Config("Sin ajuste por longitud", 1.2, 0.00),
    BM25Config("Mayor efecto de las repeticiones", 2.0, 0.75),
]
PARAM_QUERY = next(s for s in QUERIES if s.query_id == "q4")     # coincidencias en documentos de longitud y f(t,d) distintas
pq = analyze_query(PARAM_QUERY.text, stats)

runs = {}                                     # misma colección y mismas estadísticas en las tres ejecuciones
for cfg in CONFIGS:
    scorer = BM25Scorer(stats, cfg.k1, cfg.b)
    scores = scorer.scores(pq)
    runs[cfg.name] = {"scorer": scorer, "scores": dict(zip(stats.doc_ids, scores)), "pos": full_positions(scores)}

base_name = CONFIGS[0].name
matching = list(runs[base_name]["pos"])        # documentos con score > 0 en la ejecución base
positions = pd.DataFrame({
    "doc": [f"{d} · {stats.titles[stats.doc_ids.index(d)]}" for d in matching],
    "|d|": [int(stats.doc_len[stats.doc_ids.index(d)]) for d in matching],
    "|d|/avgdl": [round(stats.doc_len[stats.doc_ids.index(d)] / stats.avgdl, 2) for d in matching],
    **{f"pos (k1={cfg.k1}, b={cfg.b})": [runs[cfg.name]["pos"][d] for d in matching] for cfg in CONFIGS},
    **{f"score (k1={cfg.k1}, b={cfg.b})": [round(runs[cfg.name]["scores"][d], 3) for d in matching] for cfg in CONFIGS},
}).set_index("doc")
print(f"Consulta {PARAM_QUERY.query_id}: '{PARAM_QUERY.text}'")
display(positions)


def contributions_by_config(doc_ids: list[str]) -> pd.DataFrame:
    rows = []
    for cfg in CONFIGS:
        contrib = runs[cfg.name]["scorer"].term_contributions(pq)
        for doc_id in doc_ids:
            d = stats.doc_ids.index(doc_id)
            rows.append({"config": cfg.name, "doc": doc_id, "|d|": int(stats.doc_len[d]),
                         **{t: f"{contrib[d, j]:.3f} (f={int(stats.tf[d, stats.term_index[t]])})"
                            for j, t in enumerate(pq)}, "score": round(contrib[d].sum(), 3)})
    return pd.DataFrame(rows)


def max_shift(doc_id: str) -> int:
    return max(abs(runs[c.name]["pos"][doc_id] - runs[base_name]["pos"][doc_id]) for c in CONFIGS)


moved = sorted(matching, key=lambda d: -max_shift(d))[:2]                              # los dos que más se mueven
longest = max(matching, key=lambda d: stats.doc_len[stats.doc_ids.index(d)])            # y el más largo (¿se mueve?)
display(contributions_by_config(moved + [longest]))

Consulta q4: 'captain ahab white whale sea ship'


,|d|,|d|/avgdl,"pos (k1=1.2, b=0.75)","pos (k1=1.2, b=0.0)","pos (k1=2.0, b=0.75)","score (k1=1.2, b=0.75)","score (k1=1.2, b=0.0)","score (k1=2.0, b=0.75)"
doc,,,,,,,,
melville-moby_dick.txt · Moby Dick,218619,1.79,1,1,1,9.684,9.697,13.174
bible-kjv.txt · The King James Bible,854046,6.99,2,2,2,7.642,8.957,9.734
bryant-stories.txt · Stories to Tell to Children,46699,0.38,3,4,3,4.695,4.350,6.028
whitman-leaves.txt · Leaves of Grass,126605,1.04,4,3,4,4.680,4.693,5.989
shakespeare-hamlet.txt · Hamlet,30271,0.25,5,9,6,3.027,2.442,3.672
chesterton-ball.txt · The Ball and the Cross,82865,0.68,6,5,7,2.967,2.812,3.641
chesterton-brown.txt · The Wisdom of Father Brown,73288,0.60,7,6,5,2.791,2.747,3.715
austen-persuasion.txt · Persuasion,84167,0.69,8,7,8,2.732,2.689,3.604
edgeworth-parents.txt · The Parent's Assistant,170799,1.40,9,8,9,2.538,2.602,3.252


,config,doc,|d|,captain,ahab,white,whale,sea,ship,score
0,Base,shakespeare-hamlet.txt,30271,0.000 (f=0),0.000 (f=0),0.274 (f=4),1.871 (f=2),0.285 (f=6),0.596 (f=2),3.027
1,Base,chesterton-brown.txt,73288,1.460 (f=19),0.000 (f=0),0.306 (f=55),0.000 (f=0),0.306 (f=62),0.718 (f=18),2.791
2,Base,bible-kjv.txt,854046,1.456 (f=139),4.170 (f=94),0.285 (f=75),0.738 (f=3),0.305 (f=400),0.688 (f=71),7.642
3,Sin ajuste por longitud,shakespeare-hamlet.txt,30271,0.000 (f=0),0.000 (f=0),0.239 (f=4),1.475 (f=2),0.259 (f=6),0.470 (f=2),2.442
4,Sin ajuste por longitud,chesterton-brown.txt,73288,1.434 (f=19),0.000 (f=0),0.304 (f=55),0.000 (f=0),0.304 (f=62),0.705 (f=18),2.747
5,Sin ajuste por longitud,bible-kjv.txt,854046,1.512 (f=139),4.406 (f=94),0.305 (f=75),1.686 (f=3),0.309 (f=400),0.739 (f=71),8.957
6,Mayor efecto de las repeticiones,shakespeare-hamlet.txt,30271,0.000 (f=0),0.000 (f=0),0.348 (f=4),2.241 (f=2),0.370 (f=6),0.714 (f=2),3.672
7,Mayor efecto de las repeticiones,chesterton-brown.txt,73288,1.937 (f=19),0.000 (f=0),0.413 (f=55),0.000 (f=0),0.414 (f=62),0.951 (f=18),3.715
8,Mayor efecto de las repeticiones,bible-kjv.txt,854046,1.927 (f=139),5.448 (f=94),0.369 (f=75),0.690 (f=3),0.412 (f=400),0.888 (f=71),9.734


**Efecto de $k_1$ y $b$ (consulta q4)**

- **Sin ajuste por longitud ($b=0$).** Los documentos cortos pierden y los largos ganan. Hamlet ($0.25\times$ avgdl) baja del puesto 5 al 9 y su score cae de 3.027 a 2.442. Su aporte de `whale` ($f=2$) pasa de 1.871 a 1.475. La Biblia ($6.99\times$ avgdl) sube de 7.642 a 8.957, y su `whale` ($f=3$) pasa de 0.738 a 1.686 al dejar de penalizarse su longitud. Con $b=0.75$ un Hamlet corto con $f=2$ supera a la Biblia con $f=3$ en ese término. Con $b=0$ ocurre lo contrario.
- **Mayor efecto de las repeticiones ($k_1=2.0$).** La saturación llega más tarde, así que ganan los documentos con varios términos repetidos. Chesterton-Brown ($f=19$, 55, 62 y 18 en `captain`, `white`, `sea` y `ship`) sube del puesto 7 al 5 y su score crece un 33 % (2.791 a 3.715). Hamlet depende sobre todo de un término con $f=2$ y crece solo un 21 % (3.027 a 3.672), así que baja del 5 al 6.
- **Documentos que no se mueven.** Moby Dick (1.º) y la Biblia (2.º) mantienen su puesto en las tres ejecuciones aunque sus scores cambian. La Biblia pasa por 7.642, 8.957 y 9.734, y todos sus aportes por término suben (`ahab`: 4.170, 4.406, 5.448). Los puestos 3 y 4 (Bryant y Whitman) puntúan como máximo 4.695, 4.693 y 6.028 en cada ejecución, y la Biblia les saca siempre al menos 2.9 puntos. Moby Dick le saca a la Biblia 2.04, 0.74 y 3.44 puntos (la menor diferencia es con $b=0$), por lo que el cambio de puntuación no basta para cruzarlos.